In [6]:
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point
import numpy as np
import pandas as pd

# ----------------------------
# Step 0: Road hierarchy
# ----------------------------
road_hierarchy = {'A':4,'B':3,'C':2,'D':1}

# ----------------------------
# Step 1: Load data
# ----------------------------
buildings = gpd.read_file(r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Clean Road\Building_Centroids.shp")
roads = gpd.read_file(r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Clean Road\Clean_Road.shp")

buildings = buildings.to_crs(roads.crs)

roads['Unique_ID'] = roads['Unique_ID'].astype(str).str.strip()

# ----------------------------
# Step 2: explode multilines
# ----------------------------
roads = roads.explode(index_parts=False).reset_index(drop=True)

# ----------------------------
# Step 3: nearest road to building
# ----------------------------
def nearest_road(row):

    centroid = row.geometry
    dist = roads.geometry.distance(centroid)

    return roads.loc[dist.idxmin(),'Unique_ID']

buildings['Road_ID'] = buildings.apply(nearest_road,axis=1)

# ----------------------------
# Step 4: Find parent road
# ----------------------------
parent = {}
parent_dist = {}

for idx, road in roads.iterrows():

    geom = road.geometry

    if isinstance(geom,MultiLineString):
        geom = max(geom.geoms,key=lambda x:x.length)

    start_node = Point(geom.coords[0])

    level = road_hierarchy.get(road['ROAD_CLASS'],0)

    best_parent = None
    best_level = -1
    best_dist = None

    for j, other in roads.iterrows():

        if road['Unique_ID']==other['Unique_ID']:
            continue

        other_geom = other.geometry

        if isinstance(other_geom,MultiLineString):
            other_geom = max(other_geom.geoms,key=lambda x:x.length)

        if other_geom.distance(start_node) < 0.5:

            other_level = road_hierarchy.get(other['ROAD_CLASS'],0)

            if other_level >= level and other_level > best_level:

                best_level = other_level
                best_parent = other['Unique_ID']
                best_dist = other_geom.project(start_node)

    parent[road['Unique_ID']] = best_parent
    parent_dist[road['Unique_ID']] = best_dist

# ----------------------------
# Step 5: Compute x and y
# ----------------------------
x_dict = {}
y_dict = {}

for rid in roads['Unique_ID']:

    p = parent.get(rid)

    if p is None:
        x_dict[rid] = None
        y_dict[rid] = None
        continue

    y_dict[rid] = parent_dist.get(rid)

    pp = parent.get(p)

    if pp is None:
        x_dict[rid] = None
    else:
        x_dict[rid] = parent_dist.get(p)

# ----------------------------
# Step 6: Distance along C road (z)
# ----------------------------
def distance_along_road(row):

    centroid = row.geometry
    road_match = roads[roads['Unique_ID']==row['Road_ID']]

    if road_match.empty:
        return np.nan

    road_geom = road_match.geometry.values[0]

    if isinstance(road_geom,MultiLineString):
        road_geom = max(road_geom.geoms,key=lambda x:x.length)

    nearest = road_geom.interpolate(road_geom.project(centroid))
    dist = road_geom.project(nearest)

    return int(dist)

buildings['z'] = buildings.apply(distance_along_road,axis=1)

# ----------------------------
# Step 7: Left / Right detection
# ----------------------------
def left_right(row):

    centroid = row.geometry

    road_match = roads[roads['Unique_ID']==row['Road_ID']]

    if road_match.empty:
        return None

    road_geom = road_match.geometry.values[0]

    if isinstance(road_geom,MultiLineString):
        road_geom = max(road_geom.geoms,key=lambda x:x.length)

    d = road_geom.project(centroid)

    delta = 0.5

    p1 = road_geom.interpolate(max(0,d-delta))
    p2 = road_geom.interpolate(min(road_geom.length,d+delta))

    x1,y1 = p1.x,p1.y
    x2,y2 = p2.x,p2.y

    px,py = centroid.x,centroid.y

    cross = (x2-x1)*(py-y1)-(y2-y1)*(px-x1)

    return 'Left' if cross>0 else 'Right'

buildings['Side'] = buildings.apply(left_right,axis=1)

# ----------------------------
# Step 8: Assign house number
# ----------------------------
def assign_number(row):

    z = row['z']
    side = row['Side']

    if pd.isna(z) or side is None:
        return np.nan

    number = int(z)

    # Odd Even rule (same as before)
    if side=='Left':
        if number%2==0:
            number+=1
    else:
        if number%2==1:
            number+=1

    y = y_dict.get(row['Road_ID'])
    x = x_dict.get(row['Road_ID'])

    if x and y:
        return f"{int(x)}/{int(y)}-{number}"
    elif y:
        return f"{int(y)}-{number}"
    else:
        return f"{number}"

buildings['Final_House_Number'] = buildings.apply(assign_number,axis=1)
print(buildings.head(20))

# ----------------------------
# Step 9: Save buildings
# ----------------------------
output_building = r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Complete_Numbering\Final for 3 Road_Centroid\Complete_3Road.shp"

buildings.to_file(output_building)

print("Building numbering saved")

# ----------------------------
# Step 10: Create link lines
# ----------------------------
links = []

for idx,row in buildings.iterrows():

    pt = row.geometry

    road_geom = roads.loc[roads['Unique_ID']==row['Road_ID'],'geometry'].values[0]

    if isinstance(road_geom,MultiLineString):
        road_geom = max(road_geom.geoms,key=lambda x:x.length)

    nearest = road_geom.interpolate(road_geom.project(pt))

    line = LineString([(pt.x,pt.y),(nearest.x,nearest.y)])

    links.append(line)

links_gdf = gpd.GeoDataFrame(
    buildings[['Road_ID','Final_House_Number','z','Side']],
    geometry=links,
    crs=buildings.crs
)

# ----------------------------
# Step 11: Save links
# ----------------------------
output_links = r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Complete_Numbering\Final for 3 Road_Centroid\Complete_3Road_Link.shp"

links_gdf.to_file(output_links)

print("Link shapefile saved")

C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(


     gid    id roof_type ward_no  perimeter  area build_type  \
0   1533  None       TIN    None        NaN   NaN       None   
1    277  None       TIN    None        NaN   NaN       None   
2    278  None  Concrete    None        NaN   NaN       None   
3    279  None       TIN    None        NaN   NaN       None   
4    318  None  Concrete    None        NaN   NaN       None   
5    301  None  Concrete    None        NaN   NaN       None   
6    321  None  Concrete    None        NaN   NaN       None   
7    320  None  Concrete    None        NaN   NaN       None   
8    350  None       TIN    None        NaN   NaN       None   
9    351  None       TIN    None        NaN   NaN       None   
10   356  None  Concrete    None        NaN   NaN       None   
11   319  None       TIN    None        NaN   NaN       None   
12  1578  None       TIN    None        NaN   NaN       None   
13   376  None  Concrete    None        NaN   NaN       None   
14   360  None       TIN    None        

<ipython-input-6-0c654d4141b5>:203: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  buildings.to_file(output_building)
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Final_House_Number' to 'Final_Hous'
  ogr_write(


Building numbering saved
Link shapefile saved


<ipython-input-6-0c654d4141b5>:238: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_gdf.to_file(output_links)
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Final_House_Number' to 'Final_Hous'
  ogr_write(
